# Habit Streak Tracker — Agentic AI Demo (CSE476 CA1)

Two real tools (`log_habit`, `get_streak`) + a bonus `most_consistent`, a plan-act loop, and memory in `habits.json`.

Set `GITHUB_TOKEN` to run against `openai/gpt-4o` on GitHub Models. Without it, an offline `MockLLM` drives the same loop so the trace still shows.

In [ ]:
import os, json, importlib
# os.environ['GITHUB_TOKEN'] = 'ghp_...'   # <- paste PAT (models:read) to use the real model

import tools, agent
importlib.reload(tools); importlib.reload(agent)
print('LLM mode:', 'GitHub Models (openai/gpt-4o)' if os.environ.get('GITHUB_TOKEN') else 'offline MockLLM')

## 0. Memory — seed demo data and inspect `habits.json`

In [ ]:
tools.seed_demo_data()   # meditate = 7 days & read = 3 days (both ending yesterday), gym = broken
print(json.dumps(json.load(open(tools.HABITS_FILE)), indent=2))

## 1. Goal: "I meditated today"
Expected trace: `log_habit` → agent looks at the result → `get_streak` → picks the 7+ tier (7 seeded days + today = 8).

In [ ]:
r1 = agent.run_agent('I meditated today')

## 2. Same-day double log (edge case)
Logging again today must NOT increment the streak.

In [ ]:
r2 = agent.run_agent('I meditated today')  # streak unchanged, 'already counted'

## 3. A streak that grows, and one that broke
`read` was logged the last 3 days ending yesterday → logging today makes it 4 (momentum tier).
`gym` has a gap of several days → streak 0, gentle restart tier.

In [ ]:
r3 = agent.run_agent('I read today')
r4 = agent.run_agent('what is my gym streak?')

## 4. Reading memory back across habits
Answered entirely from dates written on earlier turns.

In [ ]:
r5 = agent.run_agent('Which habit am I most consistent with?')

## 5. Remaining edge cases, straight against the tools

In [ ]:
print('unknown habit  ->', tools.get_streak('yoga'))
print('blank name     ->', tools.log_habit('   '))
print('hallucinated tool ->', agent.dispatch('delete_everything', '{}'))
print('bad json args     ->', agent.dispatch('get_streak', '{name: meditate'))
print('extra bogus arg   ->', agent.dispatch('get_streak', '{"name": "meditate", "colour": "red"}'))

## 6. Persistence proof
Restart the kernel and run only the cell below — the streaks are still there because they came off disk.

In [ ]:
import tools
print(json.dumps(tools.most_consistent(), indent=2))